# Data center workbook vs PKG: can we find it in our own data?

Three questions, answered from the PKG tables in cdi_tm (read-only, no table is written):

- **A. Clients**: is every Power ID in the workbook a PKG customer, and does it have payment metrics and a credit row (DHE, PD)?
- **B. Counterparties**: do the names without a Power ID (XX / DCM / DCC ids) exist as counterparty entities with metrics?
- **C. Links and flows**: is each workbook link (data center ↔ second-order party) a direct payment relationship in PKG, and do the TDH amounts agree with ours?

How to run: keep the workbook next to this notebook, Run All. Results are saved under `dc_check_out/` so a rerun does not scan again (set `REUSE_SAVED = False` to force a fresh scan).
Matches by Power ID are **observed**; matches by company name are **inferred** and always shown separately.

In [ ]:
# ── 0. Setup ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")                      # ResourceWarning / DeprecationWarning / FutureWarning included
import re, html, time, json, hashlib, datetime, pickle
from pathlib import Path
import numpy as np
import pandas as pd

# compatibility shims (Cloudera PySpark 3.3 is older than the NumPy / pandas in new_default)
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp), ("bool", bool)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import HTML, display
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import openpyxl
from openpyxl.utils import get_column_letter

# ── config ──────────────────────────────────────────────────────────────────
XLSX_PATH   = "Datacenter Table Structure (v3).xlsx"
DB          = "bdahd01p_dlcdi1_cdi_tm"
T_CUST_DIM  = f"{DB}.pkg_cust_dim"                    # mdm_id ↔ customer_pwr_id / master_pwr_id
T_PAIR      = f"{DB}.pkg_cust_pair_month_v4"          # customer × dir × scope × counterparty × month
T_CUST_MET  = f"{DB}.pkg_cust_metrics_monthly_v4"     # customer payment metrics
T_CPTY_MET  = f"{DB}.pkg_cpty_metrics_monthly"        # counterparty entity metrics (ent_id = ent_l2)
T_TM        = f"{DB}.pkg_tm_cust_month"               # TM + credit block (DHE, PD) per power id × month

PAIR_MONTHS = None        # None = every month of the pair table; or ("2025-01", "2026-10") to scan fewer months
NAME_MONTHS = 12          # counterparty names are taken from the last N months of counterparty metrics
TDH_TAB     = "TDH"       # the tab whose amounts are payment flows
OUT_DIR     = "dc_check_out"
REUSE_SAVED = True        # load saved results when the inputs are unchanged
TOP_N       = 15

DEFAULT_DC_IDS = {"DCM9999999", "DCC9999999"}
REAL = "PNC Power ID"

if not Path(XLSX_PATH).exists():
    raise FileNotFoundError(f"Workbook not found at '{XLSX_PATH}'. Put it next to this notebook or fix XLSX_PATH.")
Path(OUT_DIR).mkdir(exist_ok=True)

# ── Spark session: inside the ACC limits (4g + 1g executors, 4g driver, max 512 executors) ──
spark = (SparkSession.builder
         .appName("pkg_datacenter_pkg_check")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.executor.cores", "1")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:                                                   # silence the Atlas lineage hook and executor-release noise
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
               "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
MD = StorageLevel.MEMORY_AND_DISK

# ── display: ONE consolidated HTML output per cell (long cells update the same output while they run) ──
_CSS = """<style>
.dcx h4{margin:14px 0 4px 0;font-family:sans-serif;font-size:13px}
.dcx table{border-collapse:collapse;font-size:12px;font-family:sans-serif}
.dcx th{background:#1f4e79;color:#fff;padding:3px 8px;text-align:left}
.dcx td{padding:2px 8px;border-bottom:1px solid #ddd;white-space:nowrap}
.dcx p{font-family:sans-serif;font-size:12px;margin:2px 0}
</style>"""

def _fmt(df, width=60):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if pd.api.types.is_numeric_dtype(s):
            if str(c).endswith("%"):
                out[c] = s.map(lambda x: "" if pd.isna(x) else f"{x:.1%}")
            else:
                out[c] = s.map(lambda x: "" if pd.isna(x) else
                               (f"{x:,.0f}" if (abs(x) >= 100 or float(x).is_integer()) else f"{x:,.2f}"))
        else:
            out[c] = s.map(lambda x: "" if pd.isna(x) else (str(x) if len(str(x)) <= width else str(x)[:width - 1] + "…"))
    return out

def _render(blocks):
    parts = [_CSS, '<div class="dcx">']
    for title, obj in blocks:
        parts.append(f"<h4>{html.escape(str(title))}</h4>")
        if isinstance(obj, pd.DataFrame):
            parts.append("<p>(no rows)</p>" if obj.empty else _fmt(obj).to_html(index=False, border=0, na_rep=""))
        else:
            parts.append(f"<p>{html.escape(str(obj))}</p>")
    parts.append("</div>")
    return "".join(parts)

def show(*blocks):
    display(HTML(_render(blocks)))

class Status:
    """Progress lines while a cell runs; the same output is replaced by the result tables at the end."""
    def __init__(self, title):
        self.title, self.t0, self.lines = title, time.time(), []
        self.h = display(HTML(""), display_id=True)
        self.step("started")
    def step(self, msg):
        self.lines.append(f"{time.strftime('%H:%M:%S')} · {msg}")
        self.h.update(HTML(_render([(self.title + " — running", "")]) + "".join(
            f"<p style='font:12px sans-serif;margin:1px 0'>{html.escape(l)}</p>" for l in self.lines)))
    def done(self, *blocks):
        mins = (time.time() - self.t0) / 60
        self.h.update(HTML(_render(list(blocks) + [("Run time", f"{mins:.1f} min")])))

def cached(name, sig_obj, fn, st=None):
    """Run fn() once and keep its result on disk; reuse it while the inputs (sig_obj) are unchanged."""
    sig = hashlib.md5(json.dumps(sig_obj, sort_keys=True, default=str).encode()).hexdigest()
    p = Path(OUT_DIR) / f"_{name}.pkl"
    if REUSE_SAVED and p.exists():
        try:
            saved = pickle.loads(p.read_bytes())
            if saved.get("sig") == sig:
                if st: st.step(f"{name}: loaded saved result")
                return saved["data"]
        except Exception:
            pass
    if st: st.step(f"{name}: running on Spark …")
    data = fn()
    p.write_bytes(pickle.dumps({"sig": sig, "data": data}))
    n = sum(len(v) for v in data.values()) if isinstance(data, dict) else len(data)
    if st: st.step(f"{name}: done ({n:,} rows)")
    return data

# ── workbook parsers (same rules as pkg_datacenter_workbook_explore) ────────
def _txt(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        if np.isnan(v):
            return None
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()[:10]
    s = str(v).strip()
    return s if s else None

def norm(v):
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else re.sub(r"\s+", " ", str(v)).strip().lower()

def clean_id(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().upper()
    if not s:
        return np.nan
    return s.zfill(10) if (s.isdigit() and len(s) < 10) else s

def id_kind(s):
    if not isinstance(s, str):
        return "blank"
    if s in DEFAULT_DC_IDS:
        return "Default DC"
    if re.fullmatch(r"\d{10}", s):
        return REAL
    if s.startswith("DCM"):
        return "DCM"
    if s.startswith("DCC"):
        return "DCC"
    if s.startswith("XX"):
        return "XX"
    return "other"

def parse_amt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = re.sub(r"[$,\s]", "", str(v))
    if s in ("", "-"):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

# Company-name key: upper case, brackets / punctuation / legal words removed, no spaces.
# The Python and the Spark version below apply the same steps, so keys are comparable.
_LEGAL = ["INC", "INCORPORATED", "LLC", "LP", "LLP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "NA", "SA", "GMBH"]
_LEGAL_RX = " (?:" + "|".join(_LEGAL) + ")(?= )"
def name_key(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = re.sub(r"\(.*?\)", " ", str(v).upper())
    s = s.replace(".", "").replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = " " + re.sub(r" +", " ", s) + " "
    s = re.sub(_LEGAL_RX, "", s).replace(" ", "")
    return s if s else None

def sp_key(c):
    s = F.upper(c)
    s = F.regexp_replace(s, r"\(.*?\)", " ")
    s = F.regexp_replace(s, r"\.", "")
    s = F.regexp_replace(s, "&", " AND ")
    s = F.regexp_replace(s, "[^A-Z0-9 ]", " ")
    s = F.concat(F.lit(" "), F.regexp_replace(s, " +", " "), F.lit(" "))
    s = F.regexp_replace(F.regexp_replace(s, _LEGAL_RX, ""), " ", "")
    return F.when(F.length(s) > 0, s)

def ids_df(values, col):
    """Small one-column Spark DataFrame from a Python list (built without pandas)."""
    return spark.createDataFrame([(str(v),) for v in values], f"{col} string")

show(("Session", f"Spark {spark.version} · pandas {pd.__version__} · numpy {np.__version__} · results folder: {OUT_DIR}/"))

## 1. Pre-flight: tables, columns, months
Checks every table and column this notebook needs before any long step. If a required column is missing it stops here and lists what the table really has.

In [ ]:
# ── 1. Pre-flight ───────────────────────────────────────────────────────────
NEED = {   # table: (required columns, optional columns used when present)
    T_CUST_DIM: (["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name"], []),
    T_PAIR:     (["cust", "dir", "scope", "other", "amount", "n_txn", "month"], []),
    T_CUST_MET: (["mdm_id", "month"], ["amt_all_in", "amt_all_out"]),
    T_CPTY_MET: (["ent_id", "ent_name", "month"], ["eligible", "amt_in", "amt_out", "n_cust_all", "hub_tier", "name_class"]),
    T_TM:       (["customer_pwr_id", "month", "dhe"], ["pd_grade", "pd_pct", "is_default", "past_due_days", "watchlist_exposure"]),
}
def table_cols(t):
    try:
        return [r["col_name"] for r in spark.sql(f"DESCRIBE {t}").collect()
                if r["col_name"] and not r["col_name"].startswith("#")]
    except Exception as e:
        return None
def table_months(t):
    try:
        ps = [r[0] for r in spark.sql(f"SHOW PARTITIONS {t}").collect()]
        ms = sorted({p.split("month=")[1].split("/")[0] for p in ps if "month=" in p})
        if ms:
            return ms
    except Exception:
        pass
    return sorted(str(r[0]) for r in spark.table(t).select("month").distinct().collect())

HAVE, MONTHS, rows, problems = {}, {}, [], []
for t, (req, opt) in NEED.items():
    cols = table_cols(t)
    if cols is None:
        problems.append(f"{t}: table not found or not readable")
        rows.append({"table": t.split(".")[1], "status": "NOT FOUND"})
        continue
    miss = [c for c in req if c not in cols]
    HAVE[t] = [c for c in opt if c in cols]
    MONTHS[t] = table_months(t) if "month" in cols else []
    if miss:
        problems.append(f"{t}: missing {miss}. Columns it has: {sorted(set(cols))}")
    rows.append({"table": t.split(".")[1], "status": "ok" if not miss else "MISSING COLUMNS",
                 "required missing": ", ".join(miss), "optional found": ", ".join(HAVE[t]),
                 "optional not found": ", ".join(c for c in opt if c not in cols),
                 "first month": MONTHS[t][0] if MONTHS[t] else "", "last month": MONTHS[t][-1] if MONTHS[t] else "",
                 "months": len(MONTHS[t])})
show(("1. Tables this notebook reads", pd.DataFrame(rows)))
if problems:
    raise RuntimeError("Pre-flight failed:\n" + "\n".join(problems))

PAIR_MONTH_LIST = [m for m in MONTHS[T_PAIR] if PAIR_MONTHS is None or PAIR_MONTHS[0] <= m <= PAIR_MONTHS[1]]
NAME_MONTH_LIST = MONTHS[T_CPTY_MET][-NAME_MONTHS:]
MET_MONTH_LIST  = MONTHS[T_CUST_MET][-12:]
TM_LAST         = MONTHS[T_TM][-1]

## 2. Workbook: ids, names and links
Reads the six relationship tabs (same reader as the first notebook) and lists what has to be looked up: real Power IDs, names without a Power ID, and the links between data centers and second-order parties.

In [ ]:
# ── 2. Workbook → nodes and links ───────────────────────────────────────────
def find_header(df, keys, max_rows=15):
    for i in range(min(max_rows, len(df))):
        cells = [norm(v) for v in df.iloc[i].tolist()]
        if all(any(k in c for c in cells) for k in keys):
            return i
    return None

WB, RAW = openpyxl.load_workbook(XLSX_PATH, data_only=True), {}
for ws in WB.worksheets:
    used = [(r, c) for (r, c), cell in getattr(ws, "_cells", {}).items() if cell.value is not None]
    nr, nc = (max(r for r, _ in used), max(c for _, c in used)) if used else (0, 0)
    rws = [[_txt(v) for v in r] for r in ws.iter_rows(min_row=1, max_row=nr, max_col=nc, values_only=True)] if nr else []
    RAW[ws.title] = pd.DataFrame(rws) if rws else pd.DataFrame()

FIELD_RULES = [("pnc_flag",    lambda h: "pnc flag" in h),
               ("master_id",   lambda h: "master power" in h),
               ("master_name", lambda h: "master name" in h),
               ("dc_id",       lambda h: h.startswith("datacenter customer power")),
               ("dc_name",     lambda h: h.startswith("datacenter customer name")),
               ("so_id",       lambda h: h.startswith("second order power")),
               ("so_name",     lambda h: h.startswith("second order customer name")),
               ("so_type",     lambda h: h.startswith("second order relationship type")),
               ("so_amount",   lambda h: h.startswith("second order loan amount"))]
frames = []
for tab, df in RAW.items():
    hdr = find_header(df, ["master power", "second order"])
    if hdr is None:
        continue
    heads = [norm(v) for v in df.iloc[hdr].tolist()]
    body = df.iloc[hdr + 1:].dropna(how="all")
    out = pd.DataFrame({"tab": tab, "excel_row": (body.index + 1).astype(str)}, index=body.index)
    for f, test in FIELD_RULES:
        j = next((j for j, h in enumerate(heads) if test(h)), None)
        out[f] = body.iloc[:, j] if j is not None else np.nan
    frames.append(out)
if not frames:
    raise RuntimeError("No relationship tab found in the workbook (looked for 'Master Power ID' + 'Second Order' headers).")
E = pd.concat(frames, ignore_index=True)
TABS = list(E.tab.unique())
for c in ("master_id", "dc_id", "so_id"):
    E[c] = E[c].map(clean_id)
for c in ("master_name", "dc_name", "so_name", "so_type"):
    E[c] = E[c].astype(object)
E["amount"] = E.so_amount.map(parse_amt)
E["type"]   = E.so_type.fillna("(blank)").astype(str).str.strip().str.title()

ROLES = [("Datacenter master", "master_id", "master_name"), ("Datacenter entity", "dc_id", "dc_name"),
         ("Second-order party", "so_id", "so_name")]
nd = []
for role, ic, nc_ in ROLES:
    d = E[[ic, nc_, "tab"]].dropna(subset=[ic]).rename(columns={ic: "id", nc_: "name"})
    d["role"] = role
    nd.append(d)
NODES = pd.concat(nd, ignore_index=True)
NODES["kind"] = NODES.id.map(id_kind)
NODES = NODES[NODES.kind != "Default DC"]
PIDS  = NODES[NODES.kind == REAL].copy()                       # real Power IDs → looked up by id
NAMES = NODES[NODES.kind != REAL].copy()                       # XX / DCM / DCC → looked up by name
NAMES["key"] = NAMES.name.map(name_key).map(lambda k: k if isinstance(k, str) and len(k) >= 3 else None)   # 1-2 letter keys are too weak to match on

# one row per distinct link: (tab, data center entity, second-order party, relationship type)
LK = (E.dropna(subset=["dc_id", "so_id"])
        .groupby(["tab", "master_id", "dc_id", "so_id", "type"], dropna=False, sort=False)
        .agg(master_name=("master_name", "first"), dc_name=("dc_name", "first"), so_name=("so_name", "first"),
             wb_amount=("amount", lambda s: s.sum(min_count=1)), wb_rows=("tab", "size")).reset_index())

cnt = []
for role, _, _ in ROLES:
    n = NODES[NODES.role == role].drop_duplicates("id")
    k = n.kind.value_counts()
    cnt.append({"role": role, "distinct ids": len(n), "real Power IDs (looked up by id)": int(k.get(REAL, 0)),
                "DCM / DCC (by name)": int(k.get("DCM", 0) + k.get("DCC", 0)), "XX (by name)": int(k.get("XX", 0)),
                "other": int(k.get("other", 0))})
lk = LK.groupby("tab", sort=False).agg(links=("so_id", "size"), with_amount=("wb_amount", "count")).reindex(TABS).reset_index()
show(("2a. What has to be found in PKG", pd.DataFrame(cnt)),
     ("2b. Links per tab (data center entity ↔ second-order party ↔ type)", lk.rename(columns={"with_amount": "with amount"})))

## 3. A. Clients: Power IDs in PKG
Each real Power ID is looked up in `pkg_cust_dim` both as a customer id and as a master (parent) id, then checked for payment metrics (last 12 months of `pkg_cust_metrics_monthly_v4`) and for a credit row in the latest month of `pkg_tm_cust_month` (DHE, PD).
A Power ID "found as master" brings its whole family of customers.

In [ ]:
# ── 3. A. Power IDs → pkg_cust_dim, customer metrics, TM / credit ───────────
st = Status("A. Power IDs in PKG")
pids = sorted(PIDS.id.unique())

def _idmap():
    b = F.broadcast(ids_df(pids, "pid"))
    cd = spark.table(T_CUST_DIM).select(F.col("mdm_id").cast("string").alias("mdm_id"),
                                        F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"),
                                        F.col("master_pwr_id").cast("string").alias("master_pwr_id"), "customer_name")
    x = (cd.join(b, F.col("customer_pwr_id") == F.col("pid")).withColumn("level", F.lit("customer"))
           .unionByName(cd.join(b, F.col("master_pwr_id") == F.col("pid")).withColumn("level", F.lit("master"))))
    return x.toPandas()
IDMAP = cached("idmap", [T_CUST_DIM, pids], _idmap, st)
if IDMAP.empty:
    sample = [r[0] for r in spark.table(T_CUST_DIM).where(F.col("customer_pwr_id").isNotNull()).select("customer_pwr_id").limit(5).collect()]
    raise RuntimeError(f"No workbook Power ID matched pkg_cust_dim. Workbook ids look like {pids[:3]}; "
                       f"customer_pwr_id values look like {sample}. The id format differs: tell Claude before going on.")
mdm_all = sorted(IDMAP.mdm_id.dropna().unique())
cpw_all = sorted(IDMAP.customer_pwr_id.dropna().unique())

def _cmet():
    amt = [F.sum(F.col(c).cast("double")).alias(c) for c in HAVE[T_CUST_MET]]
    return (spark.table(T_CUST_MET).where(F.col("month").isin(MET_MONTH_LIST))
                 .select(F.col("mdm_id").cast("string").alias("mdm_id"), "month", *HAVE[T_CUST_MET])
                 .join(F.broadcast(ids_df(mdm_all, "mdm_id")), "mdm_id")
                 .groupBy("mdm_id").agg(F.count("*").alias("met_months"), F.max("month").alias("met_last"), *amt).toPandas())
CMET = cached("cust_metrics", [T_CUST_MET, MET_MONTH_LIST, mdm_all], _cmet, st)

def _tm():
    num = [c for c in HAVE[T_TM] if c in ("pd_pct", "past_due_days", "watchlist_exposure")]
    txt = [c for c in HAVE[T_TM] if c in ("pd_grade", "is_default")]
    return (spark.table(T_TM).where(F.col("month") == TM_LAST)
                 .select(F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"), F.col("dhe").cast("double").alias("dhe"),
                         *[F.col(c).cast("double").alias(c) for c in num], *[F.col(c).cast("string").alias(c) for c in txt])
                 .join(F.broadcast(ids_df(cpw_all, "customer_pwr_id")), "customer_pwr_id").toPandas())
TMX = cached("tm_credit", [T_TM, TM_LAST, cpw_all], _tm, st)

# one row per workbook Power ID
g = IDMAP.groupby("pid")
PID = pd.DataFrame({"pid": pids}).set_index("pid")
PID["as_customer"] = IDMAP[IDMAP.level == "customer"].groupby("pid").mdm_id.nunique()
PID["as_master"]   = IDMAP[IDMAP.level == "master"].groupby("pid").mdm_id.nunique()
PID["n_mdm"]       = g.mdm_id.nunique()
PID["pkg_name"]    = g.customer_name.first()
m = IDMAP[["pid", "mdm_id"]].drop_duplicates().merge(CMET, on="mdm_id")
gm = m.groupby("pid")
PID["mdm_with_metrics"] = gm.mdm_id.nunique()
PID["metrics_last_month"] = gm.met_last.max()
for c in HAVE[T_CUST_MET]:
    PID[c + "_12m"] = gm[c].sum()
t = IDMAP[["pid", "customer_pwr_id"]].dropna().drop_duplicates().merge(TMX, on="customer_pwr_id")
gt = t.groupby("pid")
PID["tm_customers"] = gt.customer_pwr_id.nunique()
PID["dhe"] = gt.dhe.sum(min_count=1)
if "pd_pct" in TMX:
    PID["pd_pct_max"] = gt.pd_pct.max()
if "pd_grade" in TMX:
    PID["has_pd"] = gt.pd_grade.apply(lambda s: s.notna().any())
elif "pd_pct" in TMX:
    PID["has_pd"] = gt.pd_pct.apply(lambda s: s.notna().any())
if "is_default" in TMX:
    PID["is_default"] = gt.is_default.apply(lambda s: s.astype(str).str.lower().isin(["1", "1.0", "true", "y", "yes"]).any())
for c in ("as_customer", "as_master", "n_mdm", "mdm_with_metrics", "tm_customers"):
    PID[c] = PID[c].fillna(0).astype(int)
PID["found"] = PID.n_mdm > 0
PID = PID.reset_index()
wb_name = PIDS.drop_duplicates("id").set_index("id").name
PID.insert(1, "workbook_name", PID.pid.map(wb_name))
PID.insert(2, "roles", PID.pid.map(PIDS.groupby("id").role.agg(lambda s: " + ".join(sorted(set(s))))))
PID.insert(3, "tabs", PID.pid.map(PIDS.groupby("id").tab.agg(lambda s: " + ".join(sorted(set(s))))))
PID.to_csv(Path(OUT_DIR) / "dc_power_ids_in_pkg.csv", index=False)

def _summ(d):
    n = len(d)
    r = {"Power IDs": n, "found in PKG": int(d.found.sum()), "found %": d.found.mean() if n else np.nan,
         "as customer id": int((d.as_customer > 0).sum()), "as master id": int((d.as_master > 0).sum()),
         "not found": int((~d.found).sum()), "mdm_ids": int(IDMAP[IDMAP.pid.isin(d.pid)].mdm_id.nunique()),
         "with payment metrics (12m)": int((d.mdm_with_metrics > 0).sum()),
         "with TM / credit row": int((d.tm_customers > 0).sum()), "with DHE > 0": int((d.dhe.fillna(0) > 0).sum())}
    if "has_pd" in d:
        r["with a PD"] = int(d.has_pd.fillna(False).astype(bool).sum())
    cp = IDMAP[IDMAP.pid.isin(d.pid)].customer_pwr_id.dropna().unique()
    r["DHE, each customer once"] = TMX[TMX.customer_pwr_id.isin(cp)].dhe.sum()
    return r
a1 = pd.DataFrame([{"role": role, **_summ(PID[PID.pid.isin(PIDS[PIDS.role == role].id)])} for role, _, _ in ROLES]
                  + [{"role": "ALL (distinct)", **_summ(PID)}])
so = PIDS[PIDS.role == "Second-order party"]
a2 = pd.DataFrame([{"tab": tab, **{k: v for k, v in _summ(PID[PID.pid.isin(so[so.tab == tab].id)]).items()
                                    if k in ("Power IDs", "found in PKG", "found %", "with payment metrics (12m)", "with DHE > 0", "DHE, each customer once")}}
                   for tab in TABS])
nf = PID[~PID.found][["pid", "workbook_name", "roles", "tabs"]]
st.done(("A1. Workbook Power IDs found in pkg_cust_dim (customer metrics: last 12 months to "
         f"{MET_MONTH_LIST[-1]}; TM / credit: {TM_LAST})", a1),
        ("A2. Second-order Power IDs by tab", a2),
        (f"A3. Power IDs not found in PKG: {len(nf):,} (first {TOP_N})", nf.head(TOP_N)),
        ("Saved", f"{OUT_DIR}/dc_power_ids_in_pkg.csv: one row per Power ID with PKG name, mdm count, metrics, DHE and PD columns"))

## 4. B. Counterparties: names without a Power ID
Names carried by XX / DCM / DCC ids are matched to counterparty entities by a cleaned-name key (exact match on the key, so "Acme, Inc." = "ACME INC"; partial names are not matched). This is an inferred match.

In [ ]:
# ── 4. B. Non-PNC names → counterparty entities ─────────────────────────────
st = Status("B. Names without a Power ID as PKG counterparties")
keys = sorted(NAMES.key.dropna().unique())

def _ent():
    opt = HAVE[T_CPTY_MET]
    sel = [F.col("ent_id").cast("string").alias("ent_id"), "ent_name", "month"]
    sel += [F.col(c).cast("double").alias(c) for c in opt if c in ("amt_in", "amt_out", "n_cust_all", "hub_tier")]
    sel += [F.col(c).cast("string").alias(c) for c in opt if c in ("eligible", "name_class")]
    return (spark.table(T_CPTY_MET).where(F.col("month").isin(NAME_MONTH_LIST)).select(*sel)
                 .withColumn("key", sp_key(F.col("ent_name")))
                 .join(F.broadcast(ids_df(keys, "key")), "key").toPandas())
ENT = cached("cpty_entities", [T_CPTY_MET, NAME_MONTH_LIST, keys, _LEGAL], _ent, st)
for c in ("amt_in", "amt_out", "n_cust_all", "hub_tier"):
    if c not in ENT:
        ENT[c] = np.nan
ENT["eligible_flag"] = ENT["eligible"].astype(str).str.lower().isin(["1", "1.0", "true", "y", "yes"]) if "eligible" in ENT else False
last_nm = NAME_MONTH_LIST[-1]
ENTK = (ENT.sort_values("month").groupby(["key", "ent_id"])
           .agg(ent_name=("ent_name", "last"), months_active=("month", "nunique"), last_month=("month", "max"),
                amt_in=("amt_in", "sum"), amt_out=("amt_out", "sum"), n_cust=("n_cust_all", "max"),
                hub_tier=("hub_tier", "max"), full_metrics=("eligible_flag", "max")).reset_index())
KEYRES = ENTK.groupby("key").agg(entities=("ent_id", "nunique"), amt_in=("amt_in", "sum"), amt_out=("amt_out", "sum"),
                                 n_cust=("n_cust", "max"), full_metrics=("full_metrics", "max"),
                                 pkg_names=("ent_name", lambda s: " / ".join(s.drop_duplicates().head(2)))).reset_index()
NM = NAMES.drop_duplicates(["id", "role"]).merge(KEYRES, on="key", how="left")
NM["matched"] = NM.entities.fillna(0) > 0
NM.to_csv(Path(OUT_DIR) / "dc_names_in_pkg.csv", index=False)

def _nsum(d):
    n = len(d)
    return {"names": n, "seen as counterparty": int(d.matched.sum()), "seen %": d.matched.mean() if n else np.nan,
            "PKG entities": int(d.entities.fillna(0).sum()), "with full counterparty metrics": int(d.full_metrics.fillna(False).astype(bool).sum()),
            "dollars with PNC customers": d.amt_in.fillna(0).sum() + d.amt_out.fillna(0).sum()}
b1 = pd.DataFrame([{"role": role, "id type": k, **_nsum(NM[(NM.role == role) & (NM.kind == k)])}
                   for role, _, _ in ROLES for k in ("DCM", "DCC", "XX", "other") if ((NM.role == role) & (NM.kind == k)).any()])
b2 = pd.DataFrame([{"tab": tab, **_nsum(NAMES[NAMES.tab == tab].drop_duplicates("id").merge(KEYRES, on="key", how="left")
                                         .assign(matched=lambda d: d.entities.fillna(0) > 0))} for tab in TABS])
top = (NM[NM.matched].drop_duplicates("id").assign(usd=lambda d: d.amt_in.fillna(0) + d.amt_out.fillna(0))
         .sort_values("usd", ascending=False)[["id", "name", "role", "pkg_names", "entities", "n_cust", "usd"]].head(TOP_N)
         .rename(columns={"pkg_names": "name(s) in PKG", "n_cust": "PNC customers (max)", "usd": f"dollars, {len(NAME_MONTH_LIST)} months"}))
dcu = NM[(~NM.matched) & (NM.role != "Second-order party")].drop_duplicates("id")[["id", "name", "role", "key"]]
st.done((f"B1. Names without a Power ID seen as counterparty entities ({NAME_MONTH_LIST[0]} to {last_nm}; inferred, by name key)", b1),
        ("B2. By tab", b2),
        (f"B3. Largest matched names (top {TOP_N})", top),
        (f"B4. Non-PNC data centers NOT seen as counterparties: {len(dcu):,} (first {TOP_N})", dcu.head(TOP_N)),
        ("Saved", f"{OUT_DIR}/dc_names_in_pkg.csv: one row per non-PNC name with its match and PKG dollars"))

## 5. C. Pair-table scan
One pass over `pkg_cust_pair_month_v4`, keeping only (1) every relationship of the data centers that are PNC customers and (2) relationships between second-order PNC customers and counterparty entities named like a non-PNC data center. This is the long step.

In [ ]:
# ── 5. C. Scan the pair table once ──────────────────────────────────────────
st = Status("C. Pair-table scan")
ent_real = sorted(set(E.dc_id.dropna()[E.dc_id.dropna().map(id_kind) == REAL]))
mas_real = sorted(set(E.master_id.dropna()[E.master_id.dropna().map(id_kind) == REAL]))
so_real  = sorted(set(E.so_id.dropna()[E.so_id.dropna().map(id_kind) == REAL]))
dc_mdm    = sorted(IDMAP[IDMAP.pid.isin(set(ent_real) | set(mas_real))].mdm_id.unique())
party_mdm = sorted(IDMAP[IDMAP.pid.isin(so_real)].mdm_id.unique())
dc_keys   = set(NAMES[NAMES.role != "Second-order party"].key.dropna())
dc_ent    = sorted(ENT[ENT.key.isin(dc_keys)].ent_id.unique())
st.step(f"{len(PAIR_MONTH_LIST)} months ({PAIR_MONTH_LIST[0]} to {PAIR_MONTH_LIST[-1]}) · {len(dc_mdm):,} data center mdm_ids · "
        f"{len(party_mdm):,} second-order mdm_ids · {len(dc_ent):,} entities named like a non-PNC data center")

def _scan():
    pr = (spark.table(T_PAIR).where(F.col("month").isin(PAIR_MONTH_LIST) & F.col("scope").isin("int", "ext"))
               .select(F.col("cust").cast("string").alias("cust"), "dir", "scope", F.col("other").cast("string").alias("other"),
                       F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn"), "month"))
    T = F.lit(True)
    pr = pr.join(F.broadcast(ids_df(sorted(set(dc_mdm) | set(party_mdm)), "cust")), "cust")      # keep only our customers' rows
    a = (pr.join(F.broadcast(ids_df(dc_mdm, "cust").withColumn("is_dc", T)), "cust", "left")
           .join(F.broadcast(ids_df(party_mdm, "cust").withColumn("is_party", T)), "cust", "left")
           .join(F.broadcast(ids_df(dc_ent, "other").withColumn("o_dcent", T)), "other", "left")
           .join(F.broadcast(ids_df(party_mdm, "other").withColumn("o_party", T)), "other", "left"))
    for c in ("is_dc", "is_party", "o_dcent", "o_party"):
        a = a.withColumn(c, F.coalesce(F.col(c), F.lit(False)))
    hits = a.where(F.col("is_dc") | (F.col("is_party") & F.col("o_dcent") & (F.col("scope") == "ext"))).persist(MD)
    n = hits.count()
    st.step(f"scan finished: {n:,} pair-month rows kept")
    nbr = (hits.where("is_dc").groupBy("cust", "dir", "scope", "other")
               .agg(F.sum("amount").alias("amount"), F.sum("n_txn").alias("n_txn"), F.count("*").alias("n_months"),
                    F.min("month").alias("first_month"), F.max("month").alias("last_month")).toPandas())
    int_m = (hits.where(F.col("is_dc") & (F.col("scope") == "int") & F.col("o_party"))
                 .select("cust", "other", "dir", "month", "amount", "n_txn").toPandas())
    party_m = (hits.where(F.col("is_party") & F.col("o_dcent") & (F.col("scope") == "ext"))
                   .select("cust", "other", "dir", "month", "amount", "n_txn").toPandas())
    hits.unpersist()
    return {"nbr": nbr, "int_m": int_m, "party_m": party_m}
SCAN = cached("pair_scan", [T_PAIR, PAIR_MONTH_LIST, dc_mdm, party_mdm, dc_ent], _scan, st)
NBR, INT_M, PARTY_M = SCAN["nbr"], SCAN["int_m"], SCAN["party_m"]
for d in (NBR, INT_M, PARTY_M):
    for c in ("cust", "other", "dir"):
        d[c] = d[c].astype(object)

dcn = NBR.groupby("cust").agg(neighbours=("other", "nunique"), dollars=("amount", "sum"))
c1 = pd.DataFrame([
    {"item": "data center mdm_ids with at least one payment relationship", "value": f"{len(dcn):,} of {len(dc_mdm):,}"},
    {"item": "relationships of those data centers (customer × direction × counterparty)", "value": f"{len(NBR):,}"},
    {"item": "   with other PNC customers (scope int)", "value": f"{int((NBR.scope == 'int').sum()):,}"},
    {"item": "   with external counterparty entities (scope ext)", "value": f"{int((NBR.scope == 'ext').sum()):,}"},
    {"item": "dollars on those relationships, all months", "value": f"{NBR.amount.sum():,.0f}"},
    {"item": "monthly rows: data center ↔ second-order PNC customer", "value": f"{len(INT_M):,}"},
    {"item": "monthly rows: second-order PNC customer ↔ entity named like a non-PNC data center", "value": f"{len(PARTY_M):,}"},
])
st.done(("C1. What the scan kept", c1))

## 6. C. Are the workbook links payment relationships in PKG?
Each link is tested at two levels: the data center entity itself, and its parent's whole family (when the entity shows nothing).
Direction is read from the data center's side: a Customer should pay the data center, a Supplier should be paid by it.

In [ ]:
# ── 6. C. Workbook links vs PKG payment relationships ───────────────────────
COLS = ["dc_key", "so_id", "dir", "amount", "n_txn", "first_month", "last_month", "basis"]
so_map  = (IDMAP[IDMAP.pid.isin(so_real)][["pid", "mdm_id"]].drop_duplicates()
           .rename(columns={"pid": "so_id", "mdm_id": "other"}))
ent2key = ENT[["ent_id", "key"]].drop_duplicates().rename(columns={"ent_id": "other"})
so_nm   = (NAMES[NAMES.role == "Second-order party"][["id", "key"]].dropna().drop_duplicates().rename(columns={"id": "so_id"}))
dc_e    = IDMAP[IDMAP.pid.isin(ent_real)][["pid", "mdm_id"]].drop_duplicates().rename(columns={"pid": "dc_key", "mdm_id": "cust"})
dc_p    = IDMAP[IDMAP.pid.isin(mas_real)][["pid", "mdm_id"]].drop_duplicates().rename(columns={"pid": "dc_key", "mdm_id": "cust"})
dcn_e   = NAMES[NAMES.role == "Datacenter entity"][["id", "key"]].dropna().drop_duplicates().rename(columns={"id": "dc_key"})
dcn_p   = NAMES[NAMES.role == "Datacenter master"][["id", "key"]].dropna().drop_duplicates().rename(columns={"id": "dc_key"})

# data center is a PNC customer: its relationships, with the other side recognised by id or by name
F1 = NBR[NBR.scope == "int"].merge(so_map, on="other").assign(basis="ids")
F2 = NBR[NBR.scope == "ext"].merge(ent2key, on="other").merge(so_nm, on="key").assign(basis="party name")
FA = pd.concat([F1, F2], ignore_index=True)
# data center is NOT a PNC customer: seen from the second-order customer's side, flipped to the data center's view
PM = PARTY_M.merge(so_map.rename(columns={"other": "cust"}), on="cust").merge(ent2key, on="other")
PM["dir"] = PM.dir.map({"in": "out", "out": "in"})

def roll(dcmap, dcnames):
    a = (FA.merge(dcmap, on="cust").groupby(["dc_key", "so_id", "dir", "basis"])
           .agg(amount=("amount", "sum"), n_txn=("n_txn", "sum"), first_month=("first_month", "min"),
                last_month=("last_month", "max")).reset_index())
    b = (PM.merge(dcnames, on="key").groupby(["dc_key", "so_id", "dir"])
           .agg(amount=("amount", "sum"), n_txn=("n_txn", "sum"), first_month=("month", "min"),
                last_month=("month", "max")).reset_index().assign(basis="data center name"))
    return pd.concat([a[COLS], b[COLS]], ignore_index=True)

def wide(fl, key, sfx):
    cols = [key, "so_id", "pkg_in" + sfx, "pkg_out" + sfx, "n_txn" + sfx, "first_month" + sfx, "last_month" + sfx, "basis" + sfx]
    if fl.empty:
        return pd.DataFrame(columns=cols)
    g = fl.groupby(["dc_key", "so_id"])
    w = pd.DataFrame({"pkg_in" + sfx: fl[fl.dir == "in"].groupby(["dc_key", "so_id"]).amount.sum(),
                      "pkg_out" + sfx: fl[fl.dir == "out"].groupby(["dc_key", "so_id"]).amount.sum()})
    w = w.reindex(g.size().index)
    w["n_txn" + sfx] = g.n_txn.sum()
    w["first_month" + sfx] = g.first_month.min()
    w["last_month" + sfx] = g.last_month.max()
    w["basis" + sfx] = g.basis.agg(lambda s: " + ".join(sorted(set(s))))
    return w.reset_index().rename(columns={"dc_key": key})[cols]

FLOW_E, FLOW_P = roll(dc_e, dcn_e), roll(dc_p, dcn_p)
L = (LK.merge(wide(FLOW_E, "dc_id", ""), on=["dc_id", "so_id"], how="left")
       .merge(wide(FLOW_P, "master_id", "_par"), on=["master_id", "so_id"], how="left"))
for c in ("pkg_in", "pkg_out", "pkg_in_par", "pkg_out_par"):
    L[c] = pd.to_numeric(L[c], errors="coerce").fillna(0.0)

found_pid, ent_keys = set(IDMAP.pid), set(ENT.key)
def _ok(i, nm):
    return (i in found_pid) if id_kind(i) == REAL else (name_key(nm) in ent_keys)
L["dc_real"] = [(id_kind(a) == REAL) or (id_kind(b) == REAL) for a, b in zip(L.dc_id, L.master_id)]
L["so_real"] = L.so_id.map(id_kind) == REAL
L["dc_ok"]   = [_ok(a, an) or (isinstance(b, str) and _ok(b, bn)) for a, an, b, bn in zip(L.dc_id, L.dc_name, L.master_id, L.master_name)]
L["so_ok"]   = [_ok(a, an) for a, an in zip(L.so_id, L.so_name)]
L["f_ent"]   = (L.pkg_in + L.pkg_out) > 0
L["f_par"]   = (L.pkg_in_par + L.pkg_out_par) > 0

def _status(r):
    if r.dc_id in DEFAULT_DC_IDS:      return "0 Default DC bucket (no specific data center)"
    if not r.dc_real and not r.so_real: return "1 neither side is a PNC customer (cannot be in PKG)"
    if not r.dc_ok:                     return "2 data center not found in PKG"
    if not r.so_ok:                     return "3 second-order party not found in PKG"
    if r.f_ent:                         return "5 payments found"
    if r.f_par:                         return "6 payments found via the parent's family only"
    return "4 both found, no payments between them"
L["status"] = L.apply(_status, axis=1)
L["testable"] = L.status.str[0].isin(["4", "5", "6"])
L["found"] = L.status.str[0].isin(["5", "6"])
use_par = L.status.str[0] == "6"
L["m_in"], L["m_out"] = np.where(use_par, L.pkg_in_par, L.pkg_in), np.where(use_par, L.pkg_out_par, L.pkg_out)
L["m_last"]  = np.where(use_par, L.last_month_par, L.last_month)
L["m_basis"] = np.where(use_par, L.basis_par, L.basis)
exp = L.type.map({"Customer": "in", "Supplier": "out"})
L["dir_ok"] = np.where(exp == "in", L.m_in > 0, np.where(exp == "out", L.m_out > 0, np.nan))
L.loc[~L.found | exp.isna(), "dir_ok"] = np.nan

(L.drop(columns=["dc_real", "so_real", "f_ent", "f_par"])
  .to_csv(Path(OUT_DIR) / "dc_links_checked.csv", index=False))

def _row(d):
    t, f = int(d.testable.sum()), int(d.found.sum())
    dk = d.dir_ok.dropna()
    return {"links": len(d), "testable": t, "payments found": f, "found %": f / t if t else np.nan,
            "of which via parent only": int((d.status.str[0] == "6").sum()),
            "by ids (observed)": int((d.found & (d.m_basis == "ids")).sum()),
            "by name (inferred)": int((d.found & (d.m_basis != "ids")).sum()),
            "stated direction %": dk.mean() if len(dk) else np.nan}
c2 = pd.DataFrame([{"tab": tab, **_row(L[L.tab == tab])} for tab in TABS] + [{"tab": "ALL TABS", **_row(L)}])
c3 = L.groupby(["status", "tab"]).size().unstack(fill_value=0).reindex(columns=TABS, fill_value=0).rename_axis(None, axis=1)
c3.insert(0, "status", [i[2:] for i in c3.index])
c3["ALL"] = c3[TABS].sum(axis=1)
c4 = (L[L.found].groupby("type").agg(links=("so_id", "size"), pkg_in=("m_in", "sum"), pkg_out=("m_out", "sum")).reset_index()
        .rename(columns={"type": "relationship type", "pkg_in": "PKG dollars paid TO the data center", "pkg_out": "PKG dollars paid BY the data center"}))
show((f"C2. Workbook links that are direct payment relationships in PKG ({PAIR_MONTH_LIST[0]} to {PAIR_MONTH_LIST[-1]})", c2),
     ("C3. Status of every link, by tab", c3.reset_index(drop=True)),
     ("C4. Found links by relationship type: which way the money goes in PKG", c4),
     ("Saved", f"{OUT_DIR}/dc_links_checked.csv: every workbook link with its PKG dollars in / out, last month, match basis and status"))

## 7. C. TDH amounts against PKG dollars
TDH does not say which period its amounts cover, so PKG dollars are summed over many candidate periods (calendar years, and 1 / 3 / 6 / 12-month windows ending in each recent month) and the best-fitting periods are listed.
Only links found by Power ID on the party side are compared; the data center side may be matched by id or by name.

In [ ]:
# ── 7. C. TDH amounts vs PKG dollars ────────────────────────────────────────
T = L[(L.tab == TDH_TAB) & L.wb_amount.notna() & (L.wb_amount > 0)].copy()
T["exp"] = T.type.map({"Customer": "in", "Supplier": "out"})
tdh_tot = T.wb_amount.sum()
d1 = pd.DataFrame([{"TDH links with an amount": len(T), "TDH dollars": tdh_tot,
                    "links found in PKG": int(T.found.sum()), "found %": T.found.mean() if len(T) else np.nan,
                    "TDH dollars on found links": T.loc[T.found, "wb_amount"].sum(),
                    "dollar %": T.loc[T.found, "wb_amount"].sum() / tdh_tot if tdh_tot else np.nan,
                    "found in the stated direction %": T.dir_ok.dropna().mean() if T.dir_ok.notna().any() else np.nan}])
d1b = (T.assign(dc=np.where(T.dc_id.map(id_kind) == REAL, "data center has a Power ID", "data center without a Power ID (by name)"))
         .groupby(["dc", "type"]).agg(links=("so_id", "size"), found=("found", "sum"), tdh=("wb_amount", "sum"),
                                      tdh_found=("wb_amount", lambda s: s[T.loc[s.index, "found"]].sum())).reset_index())
d1b["found %"] = d1b.found / d1b.links
d1b["dollar %"] = d1b.tdh_found / d1b.tdh
d1b = d1b.rename(columns={"dc": "data center", "type": "relationship type", "tdh": "TDH dollars", "tdh_found": "TDH dollars found"})

# monthly PKG dollars per TDH link, in the stated direction, at the level the link was found
IM = INT_M.merge(so_map, on="other")
def monthly(dcmap, dcnames):
    a = IM.merge(dcmap, on="cust")[["dc_key", "so_id", "dir", "month", "amount"]]
    b = PM.merge(dcnames, on="key")[["dc_key", "so_id", "dir", "month", "amount"]]
    return pd.concat([a, b], ignore_index=True).groupby(["dc_key", "so_id", "dir", "month"], as_index=False).amount.sum()
ME, MP = monthly(dc_e, dcn_e), monthly(dc_p, dcn_p)
Tf = T[T.found & T.exp.notna()].reset_index(drop=True)
Tf["lid"] = Tf.index
se = Tf[Tf.status.str[0] == "5"].merge(ME, left_on=["dc_id", "so_id", "exp"], right_on=["dc_key", "so_id", "dir"])
sp = Tf[Tf.status.str[0] == "6"].merge(MP, left_on=["master_id", "so_id", "exp"], right_on=["dc_key", "so_id", "dir"])
SER = pd.concat([se[["lid", "month", "amount"]], sp[["lid", "month", "amount"]]], ignore_index=True)
M = (SER.pivot_table(index="lid", columns="month", values="amount", aggfunc="sum")
        .reindex(index=Tf.lid, columns=PAIR_MONTH_LIST).fillna(0.0)) if len(SER) else pd.DataFrame(0.0, index=Tf.lid, columns=PAIR_MONTH_LIST)

wins = [("all months", PAIR_MONTH_LIST)]
for y in sorted({m[:4] for m in PAIR_MONTH_LIST}):
    wins.append((f"calendar {y}", [m for m in PAIR_MONTH_LIST if m[:4] == y]))
for end in range(len(PAIR_MONTH_LIST) - 1, max(len(PAIR_MONTH_LIST) - 19, -1), -1):
    for k in (12, 6, 3, 1):
        if end - k + 1 >= 0:
            wins.append((f"{k} months to {PAIR_MONTH_LIST[end]}", PAIR_MONTH_LIST[end - k + 1:end + 1]))
wr, tdh = [], Tf.wb_amount.to_numpy(dtype=float)
for name, ms in wins:
    pk = M[ms].sum(axis=1).to_numpy(dtype=float)
    has = pk > 0
    ratio = pk[has] / tdh[has]
    wr.append({"PKG period": name, "months": len(ms), "links with PKG dollars": int(has.sum()),
               "median PKG / TDH": float(np.median(ratio)) if has.any() else np.nan,
               "within 1%": float((np.abs(ratio - 1) <= 0.01).sum() / len(tdh)) if len(tdh) else np.nan,
               "within 10%": float((np.abs(ratio - 1) <= 0.10).sum() / len(tdh)) if len(tdh) else np.nan,
               "within 25%": float((np.abs(ratio - 1) <= 0.25).sum() / len(tdh)) if len(tdh) else np.nan,
               "PKG total / TDH total": pk.sum() / tdh.sum() if tdh.sum() else np.nan})
WR = pd.DataFrame(wr).sort_values(["within 10%", "within 25%"], ascending=False).reset_index(drop=True)
WR = WR.rename(columns={"within 1%": "exact (±1%) %", "within 10%": "within ±10% %", "within 25%": "within ±25% %"})
best = WR.iloc[0]["PKG period"] if len(WR) and len(Tf) else None
blocks = [("D1. TDH links and dollars found in PKG", d1), ("D2. By kind of data center and relationship type", d1b),
          (f"D3. Which PKG period fits the TDH amounts best ({len(Tf):,} found links; shares are of those links; top 8 of {len(WR)} periods)", WR.head(8))]
if best:
    ms = dict(wins)[best]
    Tf["pkg_best"] = M[ms].sum(axis=1).to_numpy()
    Tf["ratio"] = Tf.pkg_best / Tf.wb_amount
    Tf["gap"] = (np.log(Tf.ratio.where(Tf.ratio > 0))).abs()
    ex = (Tf.sort_values("wb_amount", ascending=False)[["dc_name", "so_name", "type", "wb_amount", "pkg_best", "ratio", "m_basis"]].head(TOP_N)
            .rename(columns={"dc_name": "data center", "so_name": "second-order party", "wb_amount": "TDH amount",
                             "pkg_best": f"PKG, {best}", "ratio": "PKG / TDH", "m_basis": "matched by"}))
    blocks.append((f"D4. Largest TDH links next to PKG dollars for the best period ({best})", ex))
    Tf[["tab", "master_id", "dc_id", "so_id", "type", "dc_name", "so_name", "wb_amount", "pkg_best", "ratio", "status", "m_basis"]] \
        .to_csv(Path(OUT_DIR) / "dc_tdh_vs_pkg.csv", index=False)
miss = (T[~T.found].sort_values("wb_amount", ascending=False)[["dc_name", "so_name", "type", "wb_amount", "status"]].head(TOP_N)
          .assign(status=lambda d: d.status.str[2:]).rename(columns={"dc_name": "data center", "so_name": "second-order party", "wb_amount": "TDH amount"}))
blocks.append((f"D5. Largest TDH links NOT found in PKG: {int((~T.found).sum()):,} links (first {TOP_N})", miss))
show(*blocks)

## 8. Summary
The headline numbers of sections A to C in one table.

In [ ]:
# ── 8. Summary ──────────────────────────────────────────────────────────────
def sh(a, b):
    return f"{a:,} of {b:,} ({a / b:.1%})" if b else "0 of 0"
allL, tst = len(L), int(L.testable.sum())
S = [
    ("A. Workbook Power IDs found in pkg_cust_dim", sh(int(PID.found.sum()), len(PID))),
    ("A. … with payment metrics in the last 12 months", sh(int((PID.mdm_with_metrics > 0).sum()), len(PID))),
    ("A. … with a TM / credit row in " + TM_LAST, sh(int((PID.tm_customers > 0).sum()), len(PID))),
    ("A. … with DHE > 0", sh(int((PID.dhe.fillna(0) > 0).sum()), len(PID))),
    ("B. Names without a Power ID seen as PKG counterparties (by name)", sh(int(NM.drop_duplicates('id').matched.sum()), NM.id.nunique())),
    ("C. Workbook links that can be tested in PKG", sh(tst, allL)),
    ("C. Testable links with payments found", sh(int(L.found.sum()), tst)),
    ("C. … matched by Power IDs on both sides (observed)", f"{int((L.found & (L.m_basis == 'ids')).sum()):,}"),
    ("C. … matched by name on one side (inferred)", f"{int((L.found & (L.m_basis != 'ids')).sum()):,}"),
    (f"C. {TDH_TAB} links found", sh(int(T.found.sum()), len(T))),
    (f"C. {TDH_TAB} dollars on found links", f"{T.loc[T.found, 'wb_amount'].sum():,.0f} of {tdh_tot:,.0f}" + (f" ({T.loc[T.found, 'wb_amount'].sum() / tdh_tot:.1%})" if tdh_tot else "")),
    (f"C. Best-fitting PKG period for {TDH_TAB} amounts", str(best) if best else "n/a"),
]
show(("8. Summary", pd.DataFrame(S, columns=["check", "result"])),
     ("Files in " + OUT_DIR + "/", " · ".join(sorted(p.name for p in Path(OUT_DIR).glob("dc_*.csv")))))